In [1]:
import os
import glob
import re
import shapefile
import topojson as tp


# ============ CONFIGURA QUI I TUOI PERCORSI ============
INPUT_DIR = r"C:\Users\Alex\Dropbox\Istat\209_isole_calore_2026\OUT\Greenpeace\Shapefile"
OUTPUT_DIR = r"C:\Users\Alex\Dropbox\Istat\209_isole_calore_2026\app_aruba\raw"
DECIMALS = 3
# =======================================================


def clean_value(value, ndigits=DECIMALS):
    if value is None:
        return None
    if isinstance(value, float):
        if value != value:  # NaN
            return None
        if value.is_integer():
            return int(value)
        return round(value, ndigits)
    return value


def read_shape_as_featurecollection(shp_path, drop_prefixes=("UHI_",)):
    reader = shapefile.Reader(shp_path)
    field_names = [f[0] for f in reader.fields[1:]]
    keep_idx = [
        i for i, name in enumerate(field_names)
        if not any(name.startswith(p) for p in drop_prefixes)
    ]
    kept_names = [field_names[i] for i in keep_idx]

    features = []
    for sr in reader.shapeRecords():
        props = {name: clean_value(sr.record[i]) for i, name in zip(keep_idx, kept_names)}
        if "PRO_COM" in props and props["PRO_COM"] is not None:
            props["PRO_COM"] = str(int(props["PRO_COM"])).zfill(6)
        if "SEZ21_ID" in props and props["SEZ21_ID"] is not None:
            props["SEZ21_ID"] = str(int(props["SEZ21_ID"]))
        if "SEZ21" in props and props["SEZ21"] is not None:
            props["SEZ21"] = int(props["SEZ21"])
        features.append({
            "type": "Feature",
            "properties": props,
            "geometry": sr.shape.__geo_interface__,
        })
    reader.close()
    return {"type": "FeatureCollection", "features": features}


def convert_one(shp_path, out_dir):
    fc = read_shape_as_featurecollection(shp_path)
    topo = tp.Topology(fc, prequantize=1e5, toposimplify=0)
    base = os.path.splitext(os.path.basename(shp_path))[0]
    out_path = os.path.join(out_dir, f"{base}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(topo.to_json())
    return out_path, len(fc["features"]), os.path.getsize(out_path) / 1024


os.makedirs(OUTPUT_DIR, exist_ok=True)
pattern = os.path.join(INPUT_DIR, "STstats_*", "STstats_*.shp")
shp_files = sorted(glob.glob(pattern))
print(f"Trovati {len(shp_files)} shapefile.\n")

for shp in shp_files:
    name = re.sub(r"^STstats_", "", os.path.splitext(os.path.basename(shp))[0])
    try:
        out, n_feat, kb = convert_one(shp, OUTPUT_DIR)
        print(f"  {name:20s} -> {os.path.basename(out):32s}  {n_feat:6d} sezioni  {kb:8.1f} KB")
    except Exception as e:
        print(f"  {name:20s} ERRORE: {e}")

print(f"\nOutput in: {OUTPUT_DIR}")

Trovati 24 shapefile.

  Ancona               -> STstats_Ancona.json                  985 sezioni     525.1 KB
  Aosta                -> STstats_Aosta.json                   466 sezioni     220.7 KB
  Bari                 -> STstats_Bari.json                   2080 sezioni     952.6 KB
  Bologna              -> STstats_Bologna.json                2607 sezioni    1268.7 KB
  Bolzano              -> STstats_Bolzano.json                 535 sezioni     294.0 KB
  Cagliari             -> STstats_Cagliari.json               1782 sezioni     915.7 KB
  Campobasso           -> STstats_Campobasso.json              548 sezioni     322.8 KB
  Catania              -> STstats_Catania.json                3052 sezioni    1308.1 KB
  Catanzaro            -> STstats_Catanzaro.json               614 sezioni     392.7 KB
  Firenze              -> STstats_Firenze.json                2731 sezioni    1361.1 KB
  Genova               -> STstats_Genova.json                 4895 sezioni    2729.3 KB
  LAquila

In [2]:
import json
import glob
import os

# Stessa cartella di output della conversione
DATA_DIR = OUTPUT_DIR

INDICATOR_LABELS = {
    "LST_2019": "LST 2019", "LST_2020": "LST 2020", "LST_2021": "LST 2021",
    "LST_2022": "LST 2022", "LST_2023": "LST 2023", "LST_2024": "LST 2024",
    "LST_2025": "LST 2025",
}
NON_INDICATOR_FIELDS = {"PRO_COM", "SEZ21", "SEZ21_ID", "P1", "P14", "P29"}

manifest = {"comuni": []}
for path in sorted(glob.glob(os.path.join(DATA_DIR, "STstats_*.json"))):
    fname = os.path.basename(path)                          # STstats_Ancona.json
    nome_comune = fname.replace("STstats_", "").replace(".json", "")

    with open(path, encoding="utf-8") as f:
        topo = json.load(f)
    obj_name = list(topo["objects"].keys())[0]
    sample_props = topo["objects"][obj_name]["geometries"][0]["properties"]
    code = sample_props.get("PRO_COM", nome_comune)

    indicators = [
        {"key": k, "label": INDICATOR_LABELS.get(k, k)}
        for k in sample_props.keys()
        if k not in NON_INDICATOR_FIELDS
    ]

    manifest["comuni"].append({
        "code": code,
        "name": nome_comune,
        "file": f"data/{fname}",
        "format": "topojson",
        "indicators": indicators,
    })
    print(f"  {nome_comune}  (codice {code}, {len(indicators)} indicatori)")

with open(os.path.join(DATA_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f"\nmanifest.json scritto in {DATA_DIR}")

  Ancona  (codice 042002, 7 indicatori)
  Aosta  (codice 007003, 7 indicatori)
  Bari  (codice 072006, 7 indicatori)
  Bologna  (codice 037006, 7 indicatori)
  Bolzano  (codice 021008, 7 indicatori)
  Cagliari  (codice 092009, 7 indicatori)
  Campobasso  (codice 070006, 7 indicatori)
  Catania  (codice 087015, 7 indicatori)
  Catanzaro  (codice 079023, 7 indicatori)
  Firenze  (codice 048017, 7 indicatori)
  Genova  (codice 010025, 7 indicatori)
  LAquila  (codice 066049, 7 indicatori)
  Messina  (codice 083048, 7 indicatori)
  Milano  (codice 015146, 7 indicatori)
  Napoli  (codice 063049, 7 indicatori)
  Palermo  (codice 082053, 7 indicatori)
  Perugia  (codice 054039, 7 indicatori)
  Potenza  (codice 076063, 7 indicatori)
  Reggio_di_Calabria  (codice 080063, 7 indicatori)
  Roma  (codice 058091, 7 indicatori)
  Torino  (codice 001272, 7 indicatori)
  Trento  (codice 022205, 7 indicatori)
  Trieste  (codice 032006, 7 indicatori)
  Venezia  (codice 027042, 7 indicatori)

manifest.jso